# From State Space Search to Mamba

# Chapter 1 — What is a State?


Before talking about State Space Models, we must answer a more fundamental question:

> **What is a state?**

A **state** is all the information necessary to completely describe a system at a given instant.

For the 8-puzzle, the state is **the entire board configuration**, not the position of a single tile.

Example:

```text
1 2 3
4 5 6
7 _ 8
```

Knowing only that *tile 5 is in the center* is insufficient because many different boards satisfy that condition.

Therefore, the state must contain **all the information required** to uniquely describe the current situation.

A state may be represented in different ways, for example

$$
s=
\begin{bmatrix}
2&8&3\\
1&6&4\\
7&\square&5
\end{bmatrix}
$$

or equivalently

$$
s=(2,8,3,1,6,4,7,0,5).
$$

The important idea is not the representation itself, but that the state completely describes the system.

# Chapter 2 — State Transition


A state alone does nothing.

The interesting question is:

> **How does the system move from one state to another?**

If

```text
2 8 3
1 6 4
7 _ 5
```

becomes

```text
2 8 3
1 6 4
7 5 _
```

something produced that change.

That mechanism is called the **transition function**.

Mathematically,

$$
s_{t+1}=T(s_t,a_t)
$$

where

- $s_t$ is the current state,
- $a_t$ is an action,
- $T$ is the transition function.

At this point we are still thinking like classical AI:

Current State → Action → Next State.

# Chapter 3 — From State to Memory


Dynamic systems ask a different question.

Instead of searching for a solution, they ask:

> **What information is required to predict the future?**

This leads to the **Markov Property**:

> The future depends only on the current state, not on the complete history.

For physical systems we replace the board configuration with a state vector

$$
x_t
$$

and the action with an external input

$$
u_t.
$$

The transition becomes

$$
x_{t+1}=f(x_t,u_t).
$$

The state is now interpreted as a **compact memory** containing everything required to predict the future.

# Chapter 4 — State Equations


The function

$$
x_{t+1}=f(x_t,u_t)
$$

is initially a black box.

It receives

- the current state,
- the current input,

and produces the next state.

At this point we do not yet know how this function is computed.

# Chapter 5 — Linear State Space Models


Suppose the dynamics can be approximated by a linear function.

Then

$$
f(x_t,u_t)=Ax_t+Bu_t
$$

and therefore

$$
x_{t+1}=Ax_t+Bu_t.
$$

A useful intuition is that this equation is simply the matrix generalization of a linear function.

Classical linear equation:

$$
y=mx+b.
$$

State equation:

$$
x_{t+1}=Ax_t+Bu_t.
$$

Clarifications:

- $A$ describes how the state evolves by itself.
- $B$ describes how the external input affects the state.

A more general affine model could be written as

$$
x_{t+1}=Ax_t+Bu_t+c,
$$

but in control theory the origin is usually shifted to an equilibrium point, eliminating the constant term.

This was one of the conceptual questions discussed during the lesson.

# Chapter 6 — Continuous Systems


Physical systems evolve continuously.

Instead of computing the next state,

we compute **how the state changes**:

$$
\dot{x}(t)=Ax(t)+Bu(t).
$$

Important clarification:

This equation **does not return the next state**.

It returns the **rate of change** of the state.

To recover the state we must integrate:

$$
x(t)=x(t_0)+\int_{t_0}^{t}\dot{x}(\tau)d\tau.
$$

Therefore,

- Discrete equation → directly computes the next state.
- Continuous equation → computes the derivative.
- Integration recovers the state.

This distinction is fundamental before studying S4.

# Chapter 7 — Discretization


Computers process discrete samples.

Approximating

$$
\dot{x}(t)\approx\frac{x_{t+1}-x_t}{\Delta}
$$

gives Euler discretization

$$
x_{t+1}=x_t+\Delta(Ax_t+Bu_t).
$$

Interpretation:

Next State = Current State + Small Change.

# Chapter 8 — Exact Discretization


Euler accumulates approximation error.

Instead, State Space Models use the exact solution of the linear differential equation:

$$
x_{k+1}=\bar A x_k+\bar B u_k.
$$

where

$$
\bar A=e^{A\Delta},
$$

and

$$
\bar B=\left(\int_0^\Delta e^{A\tau}d\tau\right)B.
$$

This preserves the continuous dynamics much more faithfully than Euler.

# Chapter 9 — Hidden State


The state is reinterpreted once again.

Instead of representing position and velocity,

it becomes

> **a compressed memory of everything observed so far.**

Every new token or frame updates that memory through

$$
x_{t+1}=\bar A x_t+\bar B u_t.
$$

The mathematics remains the same.

Only the interpretation changes.

# Chapter 10 — Why RNNs Were Not Enough?


RNNs also maintain memory,

$$
h_t=\phi(W_hh_{t-1}+W_xx_t+b),
$$

but suffer from

- vanishing gradients,
- sequential computation.

Transformers solved long-range dependencies with attention, but require

$$
O(n^2)
$$

operations.

# Chapter 11 — The Birth of S4


S4 (Structured State Space for Sequence Modeling) asked a simple question:

> Can classical Linear State Space Models become sequence models?

Its main innovation was showing that State Space Models could efficiently process very long sequences.

However,

the dynamics were **fixed**.

Every token used the same matrices

$$
\bar A,\;\bar B.
$$

# Chapter 12 — Mamba


Mamba addresses S4's main limitation.

Instead of fixed dynamics,

the update becomes conceptually

$$
x_{k+1}=\bar A(u_k)x_k+\bar B(u_k)u_k.
$$

The matrices depend on the current input.

Therefore,

the hidden state becomes **adaptive** instead of passive.

Final historical evolution:

RNN → LSTM → Transformer → S4 → Mamba.

The key conceptual takeaway from the entire notebook is:

- Classical AI: state = configuration.
- Control Theory: state = sufficient information to predict the future.
- S4: state = memory of a sequence.
- Mamba: state = adaptive memory whose update depends on the current input.


# Chapter 13 — Selective State Space Models (Mamba)

Everything we have studied so far naturally leads to one question.

Up to S4, State Space Models already had a hidden state capable of storing information from long sequences.

The state update was

$$
x_{t+1}=\bar A x_t+\bar B u_t
$$

where

- $\bar A$ determines how the previous state evolves,
- $\bar B$ determines how the current input affects the state.

The key observation is that these matrices are **fixed**.

Regardless of whether the current token is important or irrelevant, the state is updated using exactly the same dynamics.

Mamba begins by asking a very simple question:

> **Should every input modify the memory in exactly the same way?**

Its answer is **no**.

This seemingly small change becomes the central contribution of the entire paper.

---

# 13.1 Motivation — Selection as a Means of Compression

The authors argue that the fundamental problem of sequence modeling is not simply remembering information.

It is **compressing context into a limited memory**.

Suppose we have a sequence containing 10,000 tokens.

No practical model can keep all information equally important forever.

Some form of compression is unavoidable.

Therefore, the real question becomes

> **What information deserves to be remembered?**

instead of

> **How can we remember everything?**

This perspective completely changes how we think about sequence models.

---

## Transformers

Transformers achieve excellent performance because they almost **avoid compression entirely**.

During autoregressive inference, every previous token remains accessible through the Key-Value (KV) cache.

Conceptually,

```text
Token1
Token2
Token3
...
Token10000
```

Every token remains available for future attention.

This makes Transformers extremely expressive.

However, storing the entire context also explains their computational cost.

Memory usage grows with sequence length.

Attention complexity is

$$
O(n^2)
$$

during training.

---

## Recurrent Models

Recurrent Neural Networks take the opposite approach.

Instead of storing the whole sequence,

they continuously compress it into a hidden state.

Conceptually,

```text
Hidden State
      ↓
Current Token
      ↓
Updated Hidden State
```

Only one memory vector survives.

Inference becomes extremely efficient because memory remains constant.

However,

this raises a new problem.

If irrelevant information occupies part of the hidden state,

important information may eventually disappear.

The model has no explicit mechanism for deciding what should remain in memory.

---

## The Real Problem

The paper argues that memory itself is **not** the bottleneck.

The bottleneck is

> **selection.**

A sequence model should not only remember.

It should also decide

- what to keep,
- what to ignore,
- what to forget.

This idea is summarized by one word that appears throughout the paper:

> **Selection**

---

# The Selective Copying Task

To illustrate this limitation, the paper introduces a synthetic task called the **Selective Copying Task**.

Suppose the input sequence is

```text
A
3
9
B
5
2
C
```

The objective is to remember only the letters.

A human immediately understands that

```text
3
9
5
2
```

are irrelevant.

Only

```text
A
B
C
```

need to be stored.

However,

a classical State Space Model updates its hidden state after **every single token**.

Since the transition matrices are fixed,

the model cannot distinguish between meaningful and meaningless inputs.

Every token affects the hidden state in essentially the same way.

This illustrates one of the main limitations of Linear Time-Invariant (LTI) State Space Models.

---

## Selection

Ideally,

the hidden state should behave more like

```text
A
↓

Store

3
↓

Ignore

9
↓

Ignore

B
↓

Store

5
↓

Ignore

2
↓

Ignore

C
↓

Store
```

In other words,

the model should decide

whether each token deserves to modify its memory.

S4 could not perform this type of input-dependent reasoning.

---

# 13.2 Improving State Space Models with Selection

This section introduces the main contribution of Mamba.

Until now,

the hidden state evolved according to

$$
x_{t+1}=\bar A x_t+\bar B u_t.
$$

Notice that

$\bar A$

and

$\bar B$

are fixed matrices.

The same transition dynamics are used for every token.

Mamba proposes replacing these constant dynamics with **input-dependent dynamics**.

Conceptually,

the recurrence becomes

$$
x_{t+1}
=
\bar A(u_t)x_t
+
\bar B(u_t)u_t.
$$

Now,

the matrices themselves depend on the current input.

This changes the interpretation of the hidden state completely.

The state is no longer updated uniformly.

Instead,

every token decides how strongly it should influence the memory.

---

## An Intuitive Example

Consider two words:

```text
dog
```

and

```text
the
```

The word

```text
dog
```

usually carries semantic meaning.

The word

```text
the
```

is often much less informative.

Ideally,

the hidden state should perform something like

```text
dog
↓

Large update
```

while

```text
the
↓

Small update
```

Mamba allows exactly this behavior.

Not every token modifies the memory equally.

Important tokens can produce large state updates,

while unimportant tokens can be almost ignored.

---

# Learning the Selection Mechanism

Instead of using fixed matrices,

Mamba learns several functions of the input.

The paper introduces

$$
\Delta(x),
$$

$$
B(x),
$$

and

$$
C(x),
$$

which are computed directly from the current token.

These functions allow the model to decide

- how much of the previous state should be preserved,
- how much of the current input should enter the memory,
- how the hidden state contributes to the output.

Rather than using a single transition rule,

the recurrence adapts itself continuously along the sequence.

---

# From Time-Invariant to Time-Varying Dynamics

One of the most important conceptual changes introduced by Mamba is that the system is no longer **time-invariant**.

In S4,

the transition matrix looked conceptually like

```text
A

A

A

A

A
```

The same dynamics governed every position in the sequence.

In Mamba,

the transition becomes

```text
A(token₁)

A(token₂)

A(token₃)

A(token₄)
```

Every token generates its own dynamics.

This is why the paper refers to Mamba as a **time-varying State Space Model**.

The hidden state becomes **adaptive** rather than passive.

---

# Why Does This Break S4?

This improvement introduces a serious computational problem.

S4 was efficient because its recurrence could be transformed into a convolution.

The equivalence between recurrence and convolution exists only because the transition matrices remain constant.

Once

\[
A
\]

changes for every token,

there is no longer a single convolution kernel capable of representing the entire sequence.

The mathematical equivalence disappears.

As a consequence,

the efficient convolution algorithm used by S4 can no longer be applied.

---

# A New Computational Challenge

At this point,

Mamba has become much more expressive.

However,

it also appears much slower.

The recurrence must now process different transition matrices at every timestep.

If implemented naively,

the computation becomes sequential again,

eliminating the efficiency advantage that made State Space Models attractive in the first place.

Therefore,

the paper must answer a second question:

> **How can we keep adaptive state updates while remaining computationally efficient?**

---

# 13.3 Efficient Implementation — Selective Scan

This section focuses on computation rather than modeling.

The mathematical model introduced in Section 13.2 is already complete.

The remaining challenge is implementation.

The solution is called

> **Selective Scan**

Its purpose is **not** to change the mathematics of Mamba.

Instead,

it changes **how the recurrence is computed**.

The authors observe that modern GPUs possess multiple levels of memory.

Instead of materializing the entire hidden state in slow global memory,

Selective Scan performs the recurrence inside fast on-chip memory whenever possible.

To achieve this,

the implementation combines several well-known optimization techniques:

- Kernel Fusion
- Parallel Scan
- Recomputation

Together,

these techniques allow the recurrent computation to remain highly efficient while avoiding the large memory costs that would normally result from storing every intermediate hidden state.

Importantly,

Selective Scan is **an implementation algorithm**, not a new learning algorithm.

The recurrence equation remains exactly the same.

Only its execution changes.

---

# The Core Contribution of Mamba

If the entire paper had to be summarized in one sentence,

it would be the following:

> **Classical State Space Models compress a sequence into memory. Mamba teaches the model how to decide what information is worth storing in that memory.**

This is the conceptual leap introduced by Mamba.

The hidden state is no longer simply a compressed representation of the past.

It becomes an **adaptive memory** whose behavior depends on the current input.

This idea forms the foundation of all later Mamba-based architectures,

including Vision Mamba,

VideoMamba,

and recent Video Anomaly Detection models built upon selective State Space Models.